### Libraries

In [ ]:
from pathlib import Path
import pandas as pd

# 02 · Clean AirNow

### Settings

In [ ]:
pd.set_option("display.max_columns", None)

RAW = Path("/Volumes/01-2TB/Philly-Air-Quality/raw")

# Philadelphia region: Pennsylvania, New Jersey, Delaware
STATE_CODES = ["42", "34", "10"]
STATE_ABBRS = ["PA", "NJ", "DE"]

AIRNOW = RAW / "airnow"
META = AIRNOW / "metadata"

# Column names from the AirNow fact sheets (the .dat files have no header row)
DAILY_COLS = [
    "ValidDate", "AQSID", "SiteName", "ParameterName", "ReportingUnits",
    "Value", "AveragingPeriod", "DataSource",
]
DAILY_V2_COLS = DAILY_COLS + ["AQIValue", "AQICategory", "Latitude", "Longitude", "FullAQSID"]
REPORTING_AREA_COLS = [
    "IssueDate", "ValidDate", "ValidTime", "TimeZone", "RecordSequence", "DataType",
    "Primary", "ReportingArea", "StateCode", "Latitude", "Longitude", "ParameterName",
    "AQIValue", "AQICategory", "ActionDay", "DiscussionText", "ForecastSource",
]

### Functions

In [ ]:
def read_dat(path, names=None):
    """Read one pipe-delimited AirNow file as strings."""
    return pd.read_csv(
        path, sep="|", header=None, names=names, dtype=str,
        encoding="latin-1", quoting=3,
    )


def read_airnow(filename, names, keep):
    """Read filename from every day folder, keeping rows where keep(df) is True."""
    frames = []
    for path in sorted(AIRNOW.glob(f"[0-9]*/[0-9]*/{filename}")):
        df = read_dat(path, names)
        frames.append(df[keep(df)])
    return pd.concat(frames, ignore_index=True)

## Read data

In [ ]:
airnow_daily = read_airnow(
    "daily_data.dat", DAILY_COLS,
    keep=lambda df: df["AQSID"].str[:2].isin(STATE_CODES),
)

In [ ]:
airnow_daily_v2 = read_airnow(
    "daily_data_v2.dat", DAILY_V2_COLS,
    keep=lambda df: df["FullAQSID"].str[:5].isin(["840" + s for s in STATE_CODES]),
)

In [ ]:
airnow_forecast = read_airnow(
    "reportingarea.dat", REPORTING_AREA_COLS,
    keep=lambda df: df["StateCode"].isin(STATE_ABBRS),
)

In [ ]:
airnow_sites = pd.read_csv(META / "Monitoring_Site_Locations_V2.dat", sep="|", dtype=str, encoding="latin-1")

In [ ]:
airnow_site_to_area = pd.read_csv(META / "Site_To_ReportingArea.csv", dtype=str, encoding="latin-1")

In [ ]:
# No header row and no column list in the fact sheets; name these during cleaning
airnow_area_meta = read_dat(META / "reportingarea_metadata.dat")